## Day 1 Homework
- Create a new uv project in the project directory
- Select a GitHub repo with documentation (preferably with .md files)
- Download the data from there using the techniques we've learned
- Make a post on social media about what you're building

In [1]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data

In [2]:
# this my personal repository that I use for storing data engineering notes
learning_data_engineering_docs = read_repo_data('Joyan9', 'learning_data_engineering')
print(f"LDE documents: {len(learning_data_engineering_docs)}")

LDE documents: 47


## Day 2 Homework
- For the project you selected, apply chunking
- Experiment with simple chunking, paragraph chunking + sliding window, and section chunking
- Which approach makes sense for your application? Manually inspect the results and analyze what works best
- Make a post on social media about what you're building

In [3]:
total_lenght_of_content = 0
number_of_docs = len(learning_data_engineering_docs)
for i,doc in enumerate(learning_data_engineering_docs):
    total_lenght_of_content += len(doc['content'])
    print(f"Lenght of content in doc #{i}: {len(doc['content'])}")


print(f"Avg Lenght of Content: {total_lenght_of_content/number_of_docs}")

Lenght of content in doc #0: 3023
Lenght of content in doc #1: 3977
Lenght of content in doc #2: 27069
Lenght of content in doc #3: 8543
Lenght of content in doc #4: 12114
Lenght of content in doc #5: 5280
Lenght of content in doc #6: 4599
Lenght of content in doc #7: 4313
Lenght of content in doc #8: 4066
Lenght of content in doc #9: 2279
Lenght of content in doc #10: 1662
Lenght of content in doc #11: 4470
Lenght of content in doc #12: 4599
Lenght of content in doc #13: 7844
Lenght of content in doc #14: 7950
Lenght of content in doc #15: 1967
Lenght of content in doc #16: 5234
Lenght of content in doc #17: 4435
Lenght of content in doc #18: 4027
Lenght of content in doc #19: 3362
Lenght of content in doc #20: 8914
Lenght of content in doc #21: 2399
Lenght of content in doc #22: 4289
Lenght of content in doc #23: 12426
Lenght of content in doc #24: 8167
Lenght of content in doc #25: 12507
Lenght of content in doc #26: 8022
Lenght of content in doc #27: 7735
Lenght of content in doc #

In [20]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [23]:
lde_chunks = []

for doc in learning_data_engineering_docs:
    # create a shollow copy of each doc
    doc_copy = doc.copy()
    # the pop method removes the content key and returns the value
    doc_content = doc_copy.pop('content')
    # each chunk still has other keys serving as the metadata
    chunks = sliding_window(doc_content, size=2000, step=1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    lde_chunks.extend(chunks)

In [24]:
lde_chunks

[{'start': 0,
  'chunk': "# Apache Airflow\n\n---\n\n-  It is an open-source platform for developing, scheduling, and monitoring batch-oriented workflows.\n- Workflows ar defined as python code\n\n## Reasons for Apache Airflow's Popularity\n- We know that we can use CRON job scheduler to run scripts but this is till you have 3-5 scripts, but once it starts scaling and you need more granular control then that's where you would use Apache Airflow to organise your workflows\n- Airflow let's you specifically define when to run which script/task, what to do if an error occurs etc\n\n## Airflow Architecture\n- In Airflow, workflows are represented as DAGs (Directed Acyclic Graph)\n    - Directed means that the flow is only in one direction\n    - Acyclic means that there are no loops in the flow\n    - Graph denotes the visual representation of the flow\n\n### Required Components for Minimal Airflow Installation\n\n**1. Scheduler**\n\n- Responsible for triggering workflows and submitting tas

In [28]:
len(lde_chunks)

270

It can be seen that the study notes are cut off abruptly, leading to a loss of context.

In [25]:
# splitting by paragraphs and sections

import re

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections


In [26]:
lde_chunks_split_by_level = []

for doc in learning_data_engineering_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        lde_chunks_split_by_level.append(section_doc)

lde_chunks_split_by_level

[{'filename': 'learning_data_engineering-main/Apache Airflow/Introduction to Airflow.md',
  'section': "## Reasons for Apache Airflow's Popularity\n\n- We know that we can use CRON job scheduler to run scripts but this is till you have 3-5 scripts, but once it starts scaling and you need more granular control then that's where you would use Apache Airflow to organise your workflows\n- Airflow let's you specifically define when to run which script/task, what to do if an error occurs etc"},
 {'filename': 'learning_data_engineering-main/Apache Airflow/Introduction to Airflow.md',
  'section': '## Airflow Architecture\n\n- In Airflow, workflows are represented as DAGs (Directed Acyclic Graph)\n    - Directed means that the flow is only in one direction\n    - Acyclic means that there are no loops in the flow\n    - Graph denotes the visual representation of the flow\n\n### Required Components for Minimal Airflow Installation\n\n**1. Scheduler**\n\n- Responsible for triggering workflows and

In [27]:
len(lde_chunks_split_by_level)

245

Looks like we have better results here, since none of the chunks end on an abrupt sentence, however since the notes have a lot of short paragraphs it leads to smaller chunks, not entirely useful

In [3]:
# chunking with a LLM - useful when the documentation is not structured, the simpler methods do not provide adequate results, and you have a budget

prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()

import os

from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)


def llm(prompt, model="meta-llama/llama-4-scout-17b-16e-instruct"):

    chat_completion = client.chat.completions.create(messages=[
                                                                {
                                                                    "role": "system",
                                                                    "content": "You are a helpful assistant."
                                                                },
                                                                {
                                                                    "role": "user",
                                                                    "content": prompt,
                                                                }
                                                                ],
                                                                model= model,
                                                                )

    output_text = chat_completion.choices[0].message.content

    return output_text
    

def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections

In [4]:
lde_split_by_llm = []

# processing only the first 45 documents to avoid rate limit error
for doc in learning_data_engineering_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')

    sections = intelligent_chunking(doc_content)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        lde_split_by_llm.append(section_doc)

lde_split_by_llm

[{'filename': 'learning_data_engineering-main/Apache Airflow/Introduction to Airflow.md',
  'section': '## Introduction to Apache Airflow\n\nIt is an open-source platform for developing, scheduling, and monitoring batch-oriented workflows. Workflows are defined as Python code.'},
 {'filename': 'learning_data_engineering-main/Apache Airflow/Introduction to Airflow.md',
  'section': "## Reasons for Apache Airflow's Popularity\n\nWe know that we can use CRON job scheduler to run scripts but this is till you have 3-5 scripts, but once it starts scaling and you need more granular control then that's where you would use Apache Airflow to organise your workflows. Airflow let's you specifically define when to run which script/task, what to do if an error occurs etc."},
 {'filename': 'learning_data_engineering-main/Apache Airflow/Introduction to Airflow.md',
  'section': '## Airflow Architecture\n\nIn Airflow, workflows are represented as DAGs (Directed Acyclic Graph). \n- Directed means that t

In [5]:
len(lde_split_by_llm)

191

# Day 3 Homework

- For the project you selected, index the data
- Experiment with text and vector search
- Which approach makes sense for your application? Manually inspect the results and analyze what works best
- Make a post on social media about what you're building


In [4]:
from minsearch import Index

lde_index = Index(
    text_fields = ['content', 'filename'],
    keyword_fields=[]
)

lde_index.fit(learning_data_engineering_docs)

In [8]:
query = 'Row Vs Column Oriented Databases'
results = lde_index.search(query)
results[0]

{'content': "# Row Vs Column Oriented Databases\nWe are going to mainly cover how row-oriented databases work and how column-oriented databases work.\nFollows from this video: https://youtu.be/Vw1fCeD06YI?si=w70-xS5d7wAm6Hu8\n\n### Sample table \nWe will be working with this table for understanding the difference\n\n![Sample Table](sample_table.PNG)\n\n## Row-oriented DB (Row Store)\n- The data is stored in sequential format, check the image below\n\n![row store](row_store.PNG)\n- Each row of data is stored in one block (just for illustration - in real world it could be split in multiple blocks or multiple rows can  be placed in a single block)\n\n### WHERE Clause\n- If I have a query like `SELECT * FROM sample_table WHERE id = 5`\n- Then the DBMS would need to sequentially scan all the blocks till it finds the row with `id = 5`\n- Moreover, instead of `SELECT *`, if the query was just `SELECT first_name` then we just read unnecessary amount of data - not efficient\n\n### Aggregation\n

In [9]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [10]:
record = learning_data_engineering_docs[2]
text = record['filename'] + ' ' + record['content']
v_doc = embedding_model.encode(text)

query = 'Column oriented databases'
v_query = embedding_model.encode(query)

similarity = v_query.dot(v_doc)
similarity

np.float32(0.16668737)

In [11]:
# convert my docs into embeddings
import numpy as np

lde_embeddings = []

for d in learning_data_engineering_docs:
    text = d['filename'] + ' ' + d['content']
    v = embedding_model.encode(text)
    lde_embeddings.append(v)

lde_embeddings = np.array(lde_embeddings)


In [12]:
from minsearch import VectorSearch

lde_vector_index = VectorSearch()
lde_vector_index.fit(lde_embeddings, learning_data_engineering_docs)


In [13]:
query = 'Do I have anything on column-oriented databases?'
q = embedding_model.encode(query)
results = lde_vector_index.search(q)
results

[{'content': "# Row Vs Column Oriented Databases\nWe are going to mainly cover how row-oriented databases work and how column-oriented databases work.\nFollows from this video: https://youtu.be/Vw1fCeD06YI?si=w70-xS5d7wAm6Hu8\n\n### Sample table \nWe will be working with this table for understanding the difference\n\n![Sample Table](sample_table.PNG)\n\n## Row-oriented DB (Row Store)\n- The data is stored in sequential format, check the image below\n\n![row store](row_store.PNG)\n- Each row of data is stored in one block (just for illustration - in real world it could be split in multiple blocks or multiple rows can  be placed in a single block)\n\n### WHERE Clause\n- If I have a query like `SELECT * FROM sample_table WHERE id = 5`\n- Then the DBMS would need to sequentially scan all the blocks till it finds the row with `id = 5`\n- Moreover, instead of `SELECT *`, if the query was just `SELECT first_name` then we just read unnecessary amount of data - not efficient\n\n### Aggregation\

In [14]:
query = 'Do I have anything on column-oriented databases?'

text_results = lde_index.search(query, num_results=5)

q = embedding_model.encode(query)
vector_results = lde_vector_index.search(q, num_results=5)

final_results = text_results + vector_results


In [19]:
def text_search(query):
    return lde_index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return lde_vector_index.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)
    
    # Combine and deduplicate results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)
    
    return combined_results

In [22]:
query = 'Do I have notes on analytics engineering?'

text_search_results = text_search(query)
vector_search_results = vector_search(query)
hybrid_search_results = hybrid_search(query)

In [23]:
text_search_results

[{'content': '# **Indexing Strategies**\n\nLet\'s first look at the different types of indexes\n\n1. Clustered Index\n- Sort and store the data based on key values in a table or view\n- Only 1 per table as data can be sorted in one direction only\n- It\'s like a phone book sorted alphabetically - it serves the purpose of showing the direction and at the same place giving the necessary info in this a telephone number\n- Usually used for Primary Key Columns or date columns\n\n2. Non-clustered Index\n- It is sorted list which contains the nonclustered index key values and each key value entry has a pointer to the data row\n- Does not change the way how the data is stored\n- It is more like a index in a book, it does not organise the book\n- Used for Non-PK Key columns like FKs, Filter Columns etc\n\n3. Unique Index\n- Ensures that there no duplicates or NULL value more than once\n\n4. Columnstore Index\n- Stores data by columns instead of rows, improving query performance in large dataset

In [24]:
vector_search_results

[{'content': '# Learning Data Engineering 🚀  \n\nThis repository documents my journey in learning data engineering concepts, tools, and best practices. It contains structured notes, hands-on exercises, and references to key topics in the field.  \n\n## 📂 Repository Structure  \n\n- **Apache Airflow**   \n- **Apache Spark**  \n- **Data Engineering Best Practices**  \n- **Data Engineering Fundamentals Concepts**   \n- **Data Modelling** – Dimensional modelling techniques, scenario-based examples, and interview questions.  \n- **Data Processing** – Comparison of batch vs. stream processing techniques.  \n- **Python Concepts** – Key Python programming concepts useful for data engineering.  \n- **DBT/SQL for Modularity** – SQL refactoring techniques and best practices using dbt.  \n- **dlt (data load tool)** – Open-source Python Library to load data from multiple sources, reduces boilerplate code, features like auto schema detection, unnesting etc.\n\n## 🔥 Current Focus  \n- Completing the 

In [25]:
hybrid_search_results

[{'content': '# **Indexing Strategies**\n\nLet\'s first look at the different types of indexes\n\n1. Clustered Index\n- Sort and store the data based on key values in a table or view\n- Only 1 per table as data can be sorted in one direction only\n- It\'s like a phone book sorted alphabetically - it serves the purpose of showing the direction and at the same place giving the necessary info in this a telephone number\n- Usually used for Primary Key Columns or date columns\n\n2. Non-clustered Index\n- It is sorted list which contains the nonclustered index key values and each key value entry has a pointer to the data row\n- Does not change the way how the data is stored\n- It is more like a index in a book, it does not organise the book\n- Used for Non-PK Key columns like FKs, Filter Columns etc\n\n3. Unique Index\n- Ensures that there no duplicates or NULL value more than once\n\n4. Columnstore Index\n- Stores data by columns instead of rows, improving query performance in large dataset

#### A Quick Example
| Query | Vector Search Result | Keyword Search Result |
| :--- | :--- | :--- |
| **"Acme Drill 500"** | Finds "Heavy-duty power tools" | Finds "Acme Drill 500" |
| **"Tool for making holes"** | Finds "Acme Drill 500" | Finds nothing (no match for "holes") |


# Day 4 Homework

- Do the same, but for the documentation we extracted instead of FAQ
- For the project you selected, create an agent with Pydantic AI
- Interact with the agent, improve the system prompt if needed
- Make a post on social media about what you're building


In [25]:
import os
from groq import Groq
from dotenv import load_dotenv

In [26]:
load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))


In [27]:
from typing import List, Any

def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the data engineering notes index.
    Args:
        query: The search query string.
    """
    return lde_index.search(query, num_results=5)

In [28]:
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel
from typing import List, Any

In [29]:
# 1. Define the model
model = GroqModel('llama-3.1-8b-instant')

In [30]:
# 2. Match the prompt name to the function name
system_prompt = """
You are a helpful study assistant. 
You have access to a 'text_search' tool. 
Always use this tool to look up information before answering questions.
"""

In [31]:
agent = Agent(
    model,
    system_prompt=system_prompt,
)

In [32]:
# 3. Rename this to match the prompt ('text_search')
@agent.tool_plain
def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the data engineering notes index.
    Args:
        query: The search query string.
    """
    return lde_index.search(query, num_results=5)

In [33]:
# 4. Run the agent. We use await because Pydantic AI is asynchronous
result = await agent.run("Do I have any notes on column-oriented databases?")

# 5. Use .data to see the final natural language answer
print(result.output)

Based on your text search, it appears that you have extensive notes on column-oriented databases, along with comparisons to row-oriented databases. Additionally, you have notes on cardinality estimation and its impact, as well as an in-depth look at the different data stores, including relational databases, key-value stores, document stores, graph databases, data analytics stores, column-family databases, search engine databases, time series databases, and object storage. You also have a README file that documents your journey in learning data engineering concepts, tools, and best practices.

However, your notes do not seem to cover column-oriented database specifics. You may want to check if any other notes might be relevant.


In [34]:
import json

def debug_agent_story(json_bytes):
    # Decode bytes to string and then to list
    data = json.loads(json_bytes)
    
    
    for i, message in enumerate(data):
        kind = message.get("kind", "unknown").upper()
        parts = message.get("parts", [])
        
        for part in parts:
            pk = part.get("part_kind")
            
            if pk == "system-prompt":
                print(f"[SYSTEM INSTRUCTIONS]")
                print(f"   {part.get('content').strip()}\n")
                
            elif pk == "user-prompt":
                print(f"[USER ASKS]")
                print(f"   \"{part.get('content')}\"\n")
                
            elif pk == "tool-call":
                print(f"[TOOL CALL]: {part.get('tool_name')}")
                print(f"   Args: {part.get('args')}\n")
                
            elif pk == "tool-return":
                print(f"[TOOL RESPONSE]")
                print(f"   Result: {part.get('content')}\n")
                
            elif pk == "text":
                # Check if it's a final response or just thinking out loud
                label = "[MODEL RESPONSE]" if kind == "RESPONSE" else "💭 [THOUGHT]"
                print(f"{label}")
                print(f"   {part.get('content').strip()}\n")

        # Optional: Print token usage if it exists in this turn
        if "usage" in message and message["usage"]:
            u = message["usage"]
            print(f"(Tokens Used: {u.get('input_tokens')} in / {u.get('output_tokens')} out)")
            print("---" * 10 + "\n")

# To use it:
debug_agent_story(result.all_messages_json())

[SYSTEM INSTRUCTIONS]
   You are a helpful study assistant. 
You have access to a 'text_search' tool. 
Always use this tool to look up information before answering questions.

[USER ASKS]
   "Do I have any notes on column-oriented databases?"

[TOOL CALL]: text_search
   Args: {"query":"column-oriented databases notes"}

(Tokens Used: 291 in / 17 out)
------------------------------

[TOOL RESPONSE]
   Result: [{'content': "# Row Vs Column Oriented Databases\nWe are going to mainly cover how row-oriented databases work and how column-oriented databases work.\nFollows from this video: https://youtu.be/Vw1fCeD06YI?si=w70-xS5d7wAm6Hu8\n\n### Sample table \nWe will be working with this table for understanding the difference\n\n![Sample Table](sample_table.PNG)\n\n## Row-oriented DB (Row Store)\n- The data is stored in sequential format, check the image below\n\n![row store](row_store.PNG)\n- Each row of data is stored in one block (just for illustration - in real world it could be split in 